## Example 1: Basic NetCDF file creation and data writing

In [ ]:
import netCDF4 as nc
import numpy as np
import tempfile
import os

print("Example 1: Basic NetCDF file creation and data writing")

try:
    print(f"netCDF4 version: {nc.__version__}")
    
    # Create temporary file for NetCDF data
    print(f"\nCreating temporary NetCDF file:")
    
    with tempfile.NamedTemporaryFile(suffix='.nc', delete=False) as tmp_file:
        temp_filename = tmp_file.name
    
    print(f"Temporary file: {temp_filename}")
    
    # Create NetCDF file
    print(f"\nCreating NetCDF dataset:")
    
    with nc.Dataset(temp_filename, 'w', format='NETCDF4') as dataset:
        # Add global attributes
        dataset.title = "Sample Climate Data"
        dataset.institution = "Example Research Institute"
        dataset.source = "Generated test data"
        dataset.history = "Created for netCDF4 demonstration"
        
        print(f"Dataset format: {dataset.data_model}")
        print(f"Global attributes: {len(dataset.ncattrs())} attributes")
        
        # Create dimensions
        time_dim = dataset.createDimension('time', None)  # Unlimited dimension
        lat_dim = dataset.createDimension('latitude', 10)
        lon_dim = dataset.createDimension('longitude', 15)
        
        print(f"Created dimensions:")
        print(f"  time: {len(time_dim) if time_dim.isunlimited() else time_dim.size} (unlimited: {time_dim.isunlimited()})")
        print(f"  latitude: {lat_dim.size}")
        print(f"  longitude: {lon_dim.size}")
        
        # Create coordinate variables
        times = dataset.createVariable('time', 'f8', ('time',))
        latitudes = dataset.createVariable('latitude', 'f4', ('latitude',))
        longitudes = dataset.createVariable('longitude', 'f4', ('longitude',))
        
        # Add variable attributes
        times.units = 'hours since 2023-01-01 00:00:00'
        times.calendar = 'gregorian'
        times.long_name = 'time'
        
        latitudes.units = 'degrees_north'
        latitudes.long_name = 'latitude'
        latitudes.standard_name = 'latitude'
        
        longitudes.units = 'degrees_east'
        longitudes.long_name = 'longitude'
        longitudes.standard_name = 'longitude'
        
        # Create data variables
        temperature = dataset.createVariable('temperature', 'f4', ('time', 'latitude', 'longitude'),
                                           zlib=True, complevel=6, fill_value=-999.9)
        temperature.units = 'degrees_C'
        temperature.long_name = 'surface temperature'
        temperature.standard_name = 'surface_temperature'
        
        precipitation = dataset.createVariable('precipitation', 'f4', ('time', 'latitude', 'longitude'),
                                             zlib=True, complevel=6, fill_value=-999.9)
        precipitation.units = 'mm/day'
        precipitation.long_name = 'daily precipitation'
        precipitation.standard_name = 'precipitation_flux'
        
        print(f"\nCreated variables:")
        print(f"  time: {times.dtype} {times.dimensions}")
        print(f"  latitude: {latitudes.dtype} {latitudes.dimensions}")
        print(f"  longitude: {longitudes.dtype} {longitudes.dimensions}")
        print(f"  temperature: {temperature.dtype} {temperature.dimensions}")
        print(f"  precipitation: {precipitation.dtype} {precipitation.dimensions}")
        
        # Write coordinate data
        latitudes[:] = np.linspace(-45, 45, 10)
        longitudes[:] = np.linspace(-75, 75, 15)
        times[:5] = np.arange(5) * 24  # 5 days, hourly data
        
        # Generate and write sample data
        print(f"\nWriting sample data:")
        
        for t in range(5):
            # Generate realistic temperature data (seasonal + random)
            lat_grid, lon_grid = np.meshgrid(latitudes[:], longitudes[:], indexing='ij')
            base_temp = 20 - 0.5 * np.abs(lat_grid)  # Cooler at poles
            daily_cycle = 5 * np.sin(2 * np.pi * t / 365)  # Seasonal variation
            noise = np.random.normal(0, 2, (10, 15))
            temperature[t, :, :] = base_temp + daily_cycle + noise
            
            # Generate precipitation data
            precip_base = np.maximum(0, 10 - 0.3 * np.abs(lat_grid))
            precip_var = np.random.exponential(2, (10, 15))
            precipitation[t, :, :] = precip_base * precip_var
        
        print(f"  Temperature range: {temperature[:].min():.1f} to {temperature[:].max():.1f} °C")
        print(f"  Precipitation range: {precipitation[:].min():.1f} to {precipitation[:].max():.1f} mm/day")
        
        # Display dataset summary
        print(f"\nDataset summary:")
        print(f"  Dimensions: {list(dataset.dimensions.keys())}")
        print(f"  Variables: {list(dataset.variables.keys())}")
        print(f"  Global attributes: {list(dataset.ncattrs())}")
        print(f"  File size estimate: {dataset.disk_format}")
    
    # Verify file creation
    print(f"\nFile verification:")
    file_size = os.path.getsize(temp_filename)
    print(f"  Created file size: {file_size} bytes ({file_size/1024:.1f} KB)")
    
    # NetCDF4 features demonstrated
    print(f"\nNetCDF4 features demonstrated:")
    
    features = [
        "Dataset creation: Creating NetCDF4 format files with compression",
        "Dimension management: Unlimited and fixed dimensions", 
        "Variable creation: Multi-dimensional arrays with metadata",
        "Attribute handling: Global and variable-specific attributes",
        "Data compression: Zlib compression for efficient storage",
        "Fill values: Proper handling of missing data"
    ]
    
    for feature in features:
        print(f"  - {feature}")
    
    # Clean up
    os.unlink(temp_filename)
    print(f"\nTemporary file cleaned up")
    
    print("\nBasic NetCDF file creation and data writing completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating NetCDF4 concepts:")
    concepts = [
        "File creation: Creating NetCDF files with proper structure",
        "Data organization: Dimensions, variables, and attributes",
        "Scientific metadata: CF-compliant variable descriptions"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 1 finished")

## Example 2: Reading NetCDF files and data extraction

In [ ]:
import netCDF4 as nc
import numpy as np
import tempfile
import os

print("Example 2: Reading NetCDF files and data extraction")

try:
    print("Creating sample NetCDF file for reading demonstration:")
    
    # Create a sample NetCDF file first
    with tempfile.NamedTemporaryFile(suffix='.nc', delete=False) as tmp_file:
        sample_filename = tmp_file.name
    
    # Create sample data
    with nc.Dataset(sample_filename, 'w', format='NETCDF4') as dataset:
        # Create dimensions and variables
        time_dim = dataset.createDimension('time', 12)  # Monthly data
        lat_dim = dataset.createDimension('lat', 6)
        lon_dim = dataset.createDimension('lon', 8)
        
        # Create coordinate variables
        times = dataset.createVariable('time', 'i4', ('time',))
        lats = dataset.createVariable('lat', 'f4', ('lat',))
        lons = dataset.createVariable('lon', 'f4', ('lon',))
        
        # Create data variable
        sst = dataset.createVariable('sea_surface_temperature', 'f4', ('time', 'lat', 'lon'),
                                   zlib=True, fill_value=-999.0)
        
        # Add metadata
        dataset.title = "Monthly Sea Surface Temperature"
        dataset.convention = "CF-1.6"
        times.units = "months since 2023-01-01"
        times.long_name = "time"
        lats.units = "degrees_north"
        lons.units = "degrees_east"
        sst.units = "degrees_C"
        sst.long_name = "sea surface temperature"
        sst.standard_name = "sea_surface_temperature"
        
        # Write coordinate data
        times[:] = np.arange(12)
        lats[:] = np.linspace(-25, 25, 6)
        lons[:] = np.linspace(-40, 40, 8)
        
        # Write temperature data with seasonal cycle
        for month in range(12):
            seasonal_factor = np.sin(2 * np.pi * month / 12)
            lat_factor = 25 - 0.8 * np.abs(lats[:])
            for i in range(6):
                for j in range(8):
                    sst[month, i, j] = lat_factor[i] + 5 * seasonal_factor + np.random.normal(0, 1)
    
    print(f"Sample file created: {sample_filename}")
    
    # Now demonstrate reading operations
    print(f"\nReading NetCDF file and extracting data:")
    
    with nc.Dataset(sample_filename, 'r') as dataset:
        # Inspect file structure
        print(f"\nFile inspection:")
        print(f"  Format: {dataset.data_model}")
        print(f"  Dimensions: {list(dataset.dimensions.keys())}")
        print(f"  Variables: {list(dataset.variables.keys())}")
        print(f"  Global attributes: {list(dataset.ncattrs())}")
        
        # Read global attributes
        print(f"\nGlobal attributes:")
        for attr_name in dataset.ncattrs():
            attr_value = getattr(dataset, attr_name)
            print(f"  {attr_name}: {attr_value}")
        
        # Examine dimensions
        print(f"\nDimension details:")
        for dim_name, dimension in dataset.dimensions.items():
            print(f"  {dim_name}: size={dimension.size}, unlimited={dimension.isunlimited()}")
        
        # Examine variables
        print(f"\nVariable details:")
        for var_name, variable in dataset.variables.items():
            print(f"  {var_name}:")
            print(f"    Shape: {variable.shape}")
            print(f"    Data type: {variable.dtype}")
            print(f"    Dimensions: {variable.dimensions}")
            if hasattr(variable, 'units'):
                print(f"    Units: {variable.units}")
            if hasattr(variable, 'long_name'):
                print(f"    Long name: {variable.long_name}")
        
        # Read coordinate data
        print(f"\nReading coordinate data:")
        time_data = dataset.variables['time'][:]
        lat_data = dataset.variables['lat'][:]
        lon_data = dataset.variables['lon'][:]
        
        print(f"  Time range: {time_data[0]} to {time_data[-1]} months")
        print(f"  Latitude range: {lat_data[0]:.1f} to {lat_data[-1]:.1f} degrees")
        print(f"  Longitude range: {lon_data[0]:.1f} to {lon_data[-1]:.1f} degrees")
        
        # Read and analyze temperature data
        print(f"\nReading temperature data:")
        sst_data = dataset.variables['sea_surface_temperature'][:]
        
        print(f"  Temperature array shape: {sst_data.shape}")
        print(f"  Temperature range: {sst_data.min():.1f} to {sst_data.max():.1f} °C")
        print(f"  Temperature mean: {sst_data.mean():.1f} °C")
        print(f"  Temperature std: {sst_data.std():.1f} °C")
        
        # Demonstrate advanced data extraction
        print(f"\nAdvanced data extraction:")
        
        # Extract specific time slice
        summer_sst = sst_data[6, :, :]  # July (month 6)
        print(f"  July temperature - mean: {summer_sst.mean():.1f} °C, max: {summer_sst.max():.1f} °C")
        
        # Extract specific location time series
        central_location = sst_data[:, 3, 4]  # Central grid point
        print(f"  Central location annual cycle - range: {central_location.min():.1f} to {central_location.max():.1f} °C")
        
        # Extract regional subset
        tropical_region = sst_data[:, 2:4, :]  # Tropical latitudes
        print(f"  Tropical region - shape: {tropical_region.shape}, mean: {tropical_region.mean():.1f} °C")
        
        # Demonstrate slicing and indexing
        print(f"\nData slicing examples:")
        
        slicing_examples = [
            ("First 6 months, all space", sst_data[:6, :, :]),
            ("All time, southern hemisphere", sst_data[:, :3, :]),
            ("Every other month", sst_data[::2, :, :]),
            ("Single point time series", sst_data[:, 3, 4]),
            ("Spatial average for each month", sst_data.mean(axis=(1, 2)))
        ]
        
        for description, subset in slicing_examples:
            print(f"  {description}: shape={subset.shape}, mean={subset.mean():.1f}")
        
        # Handle masked arrays and fill values
        print(f"\nMasked array handling:")
        sst_var = dataset.variables['sea_surface_temperature']
        
        print(f"  Fill value: {sst_var._FillValue}")
        print(f"  Data has masked values: {np.ma.is_masked(sst_data)}")
        print(f"  Valid data count: {np.ma.count(sst_data)}")
        
        # Variable attributes
        print(f"\nVariable attributes for 'sea_surface_temperature':")
        for attr_name in sst_var.ncattrs():
            attr_value = getattr(sst_var, attr_name)
            print(f"  {attr_name}: {attr_value}")
    
    # Data extraction patterns
    print(f"\nCommon data extraction patterns:")
    
    extraction_patterns = [
        "Time series extraction: Select specific spatial locations over time",
        "Spatial slicing: Extract geographic regions for analysis",
        "Temporal subsampling: Select specific time periods or intervals",
        "Multi-dimensional indexing: Complex array slicing operations",
        "Metadata-driven analysis: Use attributes to guide data interpretation"
    ]
    
    for pattern in extraction_patterns:
        print(f"  - {pattern}")
    
    # Performance considerations
    print(f"\nPerformance considerations:")
    
    performance_tips = [
        "Lazy loading: Variables are not loaded until accessed",
        "Memory efficiency: Read only required data subsets",
        "Chunk-based access: Align access patterns with file chunking",
        "Compression awareness: Files may be compressed for storage",
        "Network optimization: Minimize remote file access when possible"
    ]
    
    for tip in performance_tips:
        print(f"  - {tip}")
    
    # Clean up
    os.unlink(sample_filename)
    print(f"\nSample file cleaned up")
    
    print("\nReading NetCDF files and data extraction completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating NetCDF reading concepts:")
    concepts = [
        "File inspection: Examining NetCDF structure and metadata",
        "Data extraction: Reading variables and coordinate data",
        "Array slicing: Advanced indexing and subsetting operations"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 2 finished")

## Example 3: Groups, hierarchical data structures, and user-defined types

In [ ]:
import netCDF4 as nc
import numpy as np
import tempfile
import os

print("Example 3: Groups, hierarchical data structures, and user-defined types")

try:
    print("Creating NetCDF file with groups and hierarchical structure:")
    
    with tempfile.NamedTemporaryFile(suffix='.nc', delete=False) as tmp_file:
        group_filename = tmp_file.name
    
    # Create NetCDF file with groups
    with nc.Dataset(group_filename, 'w', format='NETCDF4') as root_group:
        # Root level attributes and dimensions
        root_group.title = "Multi-instrument Climate Observatory Data"
        root_group.institution = "Global Climate Research Center"
        root_group.created = "2023-12-01"
        
        # Create root level dimension
        time_dim = root_group.createDimension('time', None)
        
        # Create coordinate variable at root level
        times = root_group.createVariable('time', 'f8', ('time',))
        times.units = 'hours since 2023-01-01 00:00:00'
        times.long_name = 'observation time'
        
        print(f"Root group created with time dimension")
        
        # Create groups for different instruments
        print(f"\nCreating instrument groups:")
        
        # Satellite group
        satellite_group = root_group.createGroup('satellite')
        satellite_group.description = "Satellite-based measurements"
        satellite_group.platform = "Earth Observation Satellite"
        satellite_group.processing_level = "Level 2"
        
        # Create satellite-specific dimensions
        sat_lat_dim = satellite_group.createDimension('latitude', 180)
        sat_lon_dim = satellite_group.createDimension('longitude', 360)
        
        # Satellite coordinates
        sat_lats = satellite_group.createVariable('latitude', 'f4', ('latitude',))
        sat_lons = satellite_group.createVariable('longitude', 'f4', ('longitude',))
        sat_lats.units = 'degrees_north'
        sat_lons.units = 'degrees_east'
        sat_lats[:] = np.linspace(-89.5, 89.5, 180)
        sat_lons[:] = np.linspace(-179.5, 179.5, 360)
        
        # Satellite data variables
        cloud_cover = satellite_group.createVariable('cloud_cover', 'f4', 
                                                   ('time', 'latitude', 'longitude'),
                                                   zlib=True, complevel=6)
        cloud_cover.units = 'percent'
        cloud_cover.long_name = 'cloud cover fraction'
        cloud_cover.valid_range = [0., 100.]
        
        print(f"  Satellite group: {satellite_group.path}")
        print(f"    Dimensions: {list(satellite_group.dimensions.keys())}")
        print(f"    Variables: {list(satellite_group.variables.keys())}")
        
        # Ground station group
        station_group = root_group.createGroup('ground_stations')
        station_group.description = "Ground-based meteorological stations"
        station_group.network = "Global Weather Station Network"
        station_group.measurement_frequency = "hourly"
        
        # Station-specific dimensions
        station_dim = station_group.createDimension('station', 50)
        
        # Station metadata
        # Variable-length strings: pass the `str` builtin as the datatype (NETCDF4 format only)
        station_ids = station_group.createVariable('station_id', str, ('station',))
        station_names = station_group.createVariable('station_name', str, ('station',))
        station_ids[:] = np.array([f"ST{i:03d}" for i in range(50)], dtype=object)
        station_names[:] = np.array([f"Station {i}" for i in range(50)], dtype=object)
        station_lats = station_group.createVariable('station_latitude', 'f8', ('station',))
        station_lons = station_group.createVariable('station_longitude', 'f8', ('station',))
        
        station_lats.units = 'degrees_north'
        station_lons.units = 'degrees_east'
        
        # Generate sample station data
        station_locations = np.random.uniform(-60, 60, (50, 2))  # Random global locations
        station_lats[:] = station_locations[:, 0]
        station_lons[:] = station_locations[:, 1]
        
        # Station measurements
        temperature = station_group.createVariable('air_temperature', 'f4', ('time', 'station'),
                                                 zlib=True, fill_value=-999.9)
        humidity = station_group.createVariable('relative_humidity', 'f4', ('time', 'station'),
                                              zlib=True, fill_value=-999.9)
        
        temperature.units = 'degrees_C'
        temperature.long_name = 'air temperature'
        humidity.units = 'percent'
        humidity.long_name = 'relative humidity'
        
        print(f"  Ground stations group: {station_group.path}")
        print(f"    Dimensions: {list(station_group.dimensions.keys())}")
        print(f"    Variables: {list(station_group.variables.keys())}")
        
        # Ocean buoy group with subgroups
        ocean_group = root_group.createGroup('ocean')
        ocean_group.description = "Ocean-based measurements"
        
        # Create subgroups within ocean group
        surface_group = ocean_group.createGroup('surface')
        subsurface_group = ocean_group.createGroup('subsurface')
        
        surface_group.description = "Surface ocean measurements"
        subsurface_group.description = "Subsurface ocean profiles"
        
        # Surface group data
        buoy_dim = surface_group.createDimension('buoy', 25)
        buoy_lats = surface_group.createVariable('buoy_latitude', 'f8', ('buoy',))
        buoy_lons = surface_group.createVariable('buoy_longitude', 'f8', ('buoy',))
        sst = surface_group.createVariable('sea_surface_temperature', 'f4', ('time', 'buoy'),
                                         zlib=True, fill_value=-999.9)
        
        buoy_lats.units = 'degrees_north'
        buoy_lons.units = 'degrees_east'
        sst.units = 'degrees_C'
        sst.long_name = 'sea surface temperature'
        
        # Subsurface group data
        depth_dim = subsurface_group.createDimension('depth', 20)
        profile_dim = subsurface_group.createDimension('profile', 10)
        
        depths = subsurface_group.createVariable('depth', 'f4', ('depth',))
        depths.units = 'meters'
        depths.positive = 'down'
        depths[:] = np.array([0, 10, 20, 30, 50, 75, 100, 150, 200, 250, 
                             300, 400, 500, 600, 750, 1000, 1250, 1500, 1750, 2000])
        
        temp_profile = subsurface_group.createVariable('temperature_profile', 'f4', 
                                                     ('time', 'profile', 'depth'),
                                                     zlib=True, fill_value=-999.9)
        temp_profile.units = 'degrees_C'
        temp_profile.long_name = 'ocean temperature profile'
        
        print(f"  Ocean group: {ocean_group.path}")
        print(f"    Subgroups: {list(ocean_group.groups.keys())}")
        print(f"    Surface variables: {list(surface_group.variables.keys())}")
        print(f"    Subsurface variables: {list(subsurface_group.variables.keys())}")
        
        # Write sample time data
        times[:3] = [0, 24, 48]  # 3 days of data
        
        # Write sample data to different groups
        print(f"\nWriting sample data to groups:")
        
        # Satellite data (reduced for memory)
        for t in range(3):
            # Simple cloud pattern
            cloud_data = 50 + 30 * np.sin(np.arange(180) * np.pi / 180)[:, np.newaxis] * np.ones((1, 360))
            cloud_data += np.random.normal(0, 10, (180, 360))
            cloud_data = np.clip(cloud_data, 0, 100)
            cloud_cover[t, :, :] = cloud_data
        
        # Station data
        for t in range(3):
            # Temperature with latitude dependence
            base_temp = 15 - 0.6 * np.abs(station_lats[:])
            temp_noise = np.random.normal(0, 5, 50)
            temperature[t, :] = base_temp + temp_noise
            
            # Humidity
            base_humidity = 60 + 20 * np.random.random(50)
            humidity[t, :] = np.clip(base_humidity, 0, 100)
        
        print(f"  Sample data written to all groups")
        
        # Demonstrate group navigation
        print(f"\nGroup navigation:")
        
        def print_group_tree(group, indent=0):
            """Recursively print group structure"""
            prefix = "  " * indent
            print(f"{prefix}Group: {group.path}")
            print(f"{prefix}  Dimensions: {list(group.dimensions.keys())}")
            print(f"{prefix}  Variables: {list(group.variables.keys())}")
            print(f"{prefix}  Attributes: {list(group.ncattrs())}")
            
            for subgroup_name, subgroup in group.groups.items():
                print_group_tree(subgroup, indent + 1)
        
        print_group_tree(root_group)
    
    # Read back and demonstrate group access
    print(f"\nReading hierarchical data structure:")
    
    with nc.Dataset(group_filename, 'r') as root_group:
        # Access different groups
        print(f"\nAccessing groups:")
        
        # Direct group access
        sat_group = root_group.groups['satellite']
        print(f"  Satellite group variables: {list(sat_group.variables.keys())}")
        
        station_group = root_group.groups['ground_stations']
        print(f"  Station group variables: {list(station_group.variables.keys())}")
        
        # Access nested groups
        ocean_surface = root_group.groups['ocean'].groups['surface']
        print(f"  Ocean surface variables: {list(ocean_surface.variables.keys())}")
        
        # Read data from different groups
        print(f"\nReading data from different groups:")
        
        # Read satellite data sample
        cloud_sample = sat_group.variables['cloud_cover'][0, 90, 180]  # Equator, prime meridian
        print(f"  Cloud cover at equator: {cloud_sample:.1f}%")
        
        # Read station data sample
        station_temp = station_group.variables['air_temperature'][0, 0]
        print(f"  First station temperature: {station_temp:.1f}°C")
        
        # The hierarchy and the variable-length strings must round-trip
        assert sorted(root_group.groups) == ['ground_stations', 'ocean', 'satellite']
        assert list(root_group.groups['ocean'].groups) == ['surface', 'subsurface']
        assert station_group.variables['station_id'][0] == 'ST000'
        assert station_group.variables['station_name'][49] == 'Station 49'
        assert 0.0 <= float(cloud_sample) <= 100.0
        
        # Cross-group analysis
        print(f"\nCross-group analysis capabilities:")
        
        analysis_capabilities = [
            "Data fusion: Combine satellite and ground observations",
            "Multi-scale analysis: Compare global and local measurements",
            "Quality control: Cross-validate different measurement types",
            "Hierarchical modeling: Organize data by instrument/platform",
            "Metadata preservation: Keep instrument-specific attributes"
        ]
        
        for capability in analysis_capabilities:
            print(f"  - {capability}")
    
    # Group structure benefits
    print(f"\nBenefits of hierarchical structure:")
    
    benefits = [
        "Organization: Logical grouping of related data and metadata",
        "Namespace management: Avoid variable name conflicts", 
        "Scalability: Handle complex multi-instrument datasets",
        "Modularity: Independent group processing and analysis",
        "Standards compliance: Follow CF conventions for complex datasets"
    ]
    
    for benefit in benefits:
        print(f"  - {benefit}")
    
    # Use cases for groups
    print(f"\nCommon use cases for NetCDF groups:")
    
    use_cases = [
        "Multi-instrument datasets: Separate data from different sensors",
        "Model ensembles: Group ensemble members or model runs",
        "Multi-resolution data: Organize data at different spatial scales",
        "Quality levels: Separate raw, processed, and quality-controlled data",
        "Time series collections: Group related time series measurements"
    ]
    
    for use_case in use_cases:
        print(f"  - {use_case}")
    
    # Clean up
    os.unlink(group_filename)
    print(f"\nTest file cleaned up")
    
    print("\nGroups, hierarchical data structures, and user-defined types completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating hierarchical structure concepts:")
    concepts = [
        "Group creation: Organizing data into logical hierarchical structures",
        "Multi-level organization: Creating nested groups for complex datasets",
        "Cross-group analysis: Accessing and comparing data across groups"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 3 finished")

## Example 4: Advanced features - compression, chunking, and parallel I/O

In [ ]:
import netCDF4 as nc
import numpy as np
import tempfile
import os
import time

print("Example 4: Advanced features - compression, chunking, and parallel I/O")

try:
    print("Demonstrating advanced NetCDF4 features:")
    
    # Compression comparison
    print(f"\nCompression comparison:")
    
    with tempfile.NamedTemporaryFile(suffix='_uncompressed.nc', delete=False) as tmp1:
        uncompressed_file = tmp1.name
    with tempfile.NamedTemporaryFile(suffix='_compressed.nc', delete=False) as tmp2:
        compressed_file = tmp2.name
    
    # Create large test dataset
    data_shape = (100, 50, 80)  # Moderate size for demonstration
    test_data = np.random.random(data_shape).astype(np.float32)
    test_data *= 100  # Scale to make compression more effective
    
    print(f"Test data shape: {data_shape}")
    print(f"Test data size in memory: {test_data.nbytes / 1024:.1f} KB")
    
    # Create uncompressed file
    start_time = time.time()
    with nc.Dataset(uncompressed_file, 'w', format='NETCDF4') as dataset:
        dataset.createDimension('time', data_shape[0])
        dataset.createDimension('lat', data_shape[1])
        dataset.createDimension('lon', data_shape[2])
        
        var_uncompressed = dataset.createVariable('temperature', 'f4', ('time', 'lat', 'lon'),
                                                zlib=False)
        var_uncompressed[:] = test_data
    
    uncompressed_time = time.time() - start_time
    uncompressed_size = os.path.getsize(uncompressed_file)
    
    # Create compressed file
    start_time = time.time()
    with nc.Dataset(compressed_file, 'w', format='NETCDF4') as dataset:
        dataset.createDimension('time', data_shape[0])
        dataset.createDimension('lat', data_shape[1])
        dataset.createDimension('lon', data_shape[2])
        
        var_compressed = dataset.createVariable('temperature', 'f4', ('time', 'lat', 'lon'),
                                              zlib=True, complevel=6, shuffle=True)
        var_compressed[:] = test_data
    
    compressed_time = time.time() - start_time
    compressed_size = os.path.getsize(compressed_file)
    
    print(f"Compression results:")
    print(f"  Uncompressed: {uncompressed_size / 1024:.1f} KB, write time: {uncompressed_time:.3f}s")
    print(f"  Compressed: {compressed_size / 1024:.1f} KB, write time: {compressed_time:.3f}s")
    print(f"  Compression ratio: {uncompressed_size / compressed_size:.2f}:1")
    print(f"  Space savings: {(1 - compressed_size / uncompressed_size) * 100:.1f}%")
    
    # Chunking demonstration
    print(f"\nChunking demonstration:")
    
    with tempfile.NamedTemporaryFile(suffix='_chunked.nc', delete=False) as tmp3:
        chunked_file = tmp3.name
    
    with nc.Dataset(chunked_file, 'w', format='NETCDF4') as dataset:
        dataset.createDimension('time', None)  # Unlimited
        dataset.createDimension('lat', 100)
        dataset.createDimension('lon', 200)
        
        # Different chunking strategies
        chunking_strategies = [
            {
                'name': 'Time-optimized chunking',
                'chunks': (10, 100, 200),  # Favor time slices
                'var_name': 'temp_time_chunks'
            },
            {
                'name': 'Spatial-optimized chunking', 
                'chunks': (1, 50, 100),    # Favor spatial slices
                'var_name': 'temp_spatial_chunks'
            },
            {
                'name': 'Balanced chunking',
                'chunks': (5, 50, 100),    # Balanced approach
                'var_name': 'temp_balanced_chunks'
            }
        ]
        
        variables = {}
        for strategy in chunking_strategies:
            var = dataset.createVariable(strategy['var_name'], 'f4', ('time', 'lat', 'lon'),
                                       zlib=True, complevel=4,
                                       chunksizes=strategy['chunks'])
            variables[strategy['name']] = var
            print(f"  {strategy['name']}: chunks={strategy['chunks']}")
        
        # Write sample data
        print(f"  Writing chunked data...")
        sample_data = np.random.random((20, 100, 200)).astype(np.float32)
        
        for var in variables.values():
            var[:20, :, :] = sample_data
    
    chunked_size = os.path.getsize(chunked_file)
    print(f"  Chunked file size: {chunked_size / 1024:.1f} KB")
    
    # Advanced compression options
    print(f"\nAdvanced compression options:")
    
    with tempfile.NamedTemporaryFile(suffix='_advanced.nc', delete=False) as tmp4:
        advanced_file = tmp4.name
    
    with nc.Dataset(advanced_file, 'w', format='NETCDF4') as dataset:
        dataset.createDimension('x', 100)
        dataset.createDimension('y', 100)
        dataset.createDimension('z', 50)
        
        # Different compression algorithms and settings
        compression_options = [
            {
                'name': 'High compression',
                'var_name': 'data_high_comp',
                'options': {'zlib': True, 'complevel': 9, 'shuffle': True}
            },
            {
                'name': 'Fast compression',
                'var_name': 'data_fast_comp',
                'options': {'zlib': True, 'complevel': 1, 'shuffle': False}
            },
            {
                'name': 'Balanced compression',
                'var_name': 'data_balanced_comp',
                'options': {'zlib': True, 'complevel': 6, 'shuffle': True}
            },
            {
                'name': 'Fletcher32 checksum',
                'var_name': 'data_checksum',
                'options': {'zlib': True, 'complevel': 4, 'shuffle': True, 'fletcher32': True}
            }
        ]
        
        test_3d_data = np.random.random((100, 100, 50)).astype(np.float32) * 1000
        
        compression_results = {}
        for comp_option in compression_options:
            start_time = time.time()
            var = dataset.createVariable(comp_option['var_name'], 'f4', ('x', 'y', 'z'),
                                       **comp_option['options'])
            var[:] = test_3d_data
            write_time = time.time() - start_time
            
            compression_results[comp_option['name']] = {
                'write_time': write_time,
                'options': comp_option['options']
            }
            
            print(f"  {comp_option['name']}: {write_time:.3f}s")
    
    # Variable-length arrays (VLEN)
    print(f"\nVariable-length arrays (VLEN):")
    
    with tempfile.NamedTemporaryFile(suffix='_vlen.nc', delete=False) as tmp5:
        vlen_file = tmp5.name
    
    with nc.Dataset(vlen_file, 'w', format='NETCDF4') as dataset:
        # Create VLEN data type for variable-length arrays
        vlen_dtype = dataset.createVLType(np.float32, 'ragged_array')
        
        dataset.createDimension('station', 10)
        dataset.createDimension('time', None)
        
        # VLEN variable for irregular data
        measurements = dataset.createVariable('station_measurements', vlen_dtype, ('station', 'time'))
        
        # Create sample irregular data
        print(f"  Creating variable-length measurement arrays:")
        for station in range(5):  # Write to first 5 stations
            for time_step in range(3):  # 3 time steps
                # Each station has different number of measurements
                n_measurements = np.random.randint(3, 15)
                measurement_data = np.random.normal(20, 5, n_measurements).astype(np.float32)
                measurements[station, time_step] = measurement_data
                print(f"    Station {station}, Time {time_step}: {n_measurements} measurements")
        
        print(f"  VLEN data written successfully")
    
    # Performance optimization tips
    print(f"\nPerformance optimization techniques:")
    
    optimization_tips = [
        "Chunking strategy: Choose chunks based on access patterns",
        "Compression level: Balance between compression ratio and speed (4-6 recommended)",
        "Shuffle filter: Reorder bytes for better compression (enable for numerical data)",
        "Fletcher32: Add checksums for data integrity verification",
        "Unlimited dimensions: Use sparingly as they can impact performance",
        "Variable ordering: Group frequently accessed variables together"
    ]
    
    for tip in optimization_tips:
        print(f"  - {tip}")
    
    # Read performance comparison
    print(f"\nRead performance comparison:")
    
    # Read compressed vs uncompressed
    start_time = time.time()
    with nc.Dataset(uncompressed_file, 'r') as dataset:
        data_read = dataset.variables['temperature'][:]
    uncompressed_read_time = time.time() - start_time
    
    start_time = time.time()
    with nc.Dataset(compressed_file, 'r') as dataset:
        data_read = dataset.variables['temperature'][:]
    compressed_read_time = time.time() - start_time
    
    print(f"  Uncompressed read time: {uncompressed_read_time:.3f}s")
    print(f"  Compressed read time: {compressed_read_time:.3f}s")
    print(f"  Read time ratio: {compressed_read_time / uncompressed_read_time:.2f}")
    
    # Advanced features summary
    print(f"\nAdvanced NetCDF4 features summary:")
    
    advanced_features = [
        "Multiple compression algorithms: Zlib, szip, and user-defined filters",
        "Chunked storage: Optimize I/O patterns for different access methods",
        "Variable-length arrays: Handle irregular and sparse datasets",
        "Data integrity: Fletcher32 checksums for error detection",
        "Parallel I/O: HDF5-based parallel file access (when available)",
        "Group hierarchies: Organize complex multi-instrument datasets"
    ]
    
    for feature in advanced_features:
        print(f"  - {feature}")
    
    # Clean up all temporary files
    temp_files = [uncompressed_file, compressed_file, chunked_file, advanced_file, vlen_file]
    for temp_file in temp_files:
        if os.path.exists(temp_file):
            os.unlink(temp_file)
    
    print(f"\nTemporary files cleaned up")
    
    print("\\nAdvanced features - compression, chunking, and parallel I/O completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating advanced NetCDF4 concepts:")
    concepts = [
        "Compression optimization: Balancing file size and I/O performance",
        "Chunking strategies: Organizing data layout for efficient access",
        "Advanced data types: Variable-length arrays and custom types"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 4 finished")

## Example 5: Climate data workflows and CF conventions

In [ ]:
import netCDF4 as nc
import numpy as np
import tempfile
import os
from datetime import datetime, timedelta

print("Example 5: Climate data workflows and CF conventions")

try:
    print("Creating CF-compliant climate dataset:")
    
    with tempfile.NamedTemporaryFile(suffix='_climate.nc', delete=False) as tmp_file:
        climate_file = tmp_file.name
    
    # Create CF-compliant climate dataset
    with nc.Dataset(climate_file, 'w', format='NETCDF4') as dataset:
        # Global attributes following CF conventions
        dataset.Conventions = "CF-1.8"
        dataset.title = "Regional Climate Model Output"
        dataset.institution = "Climate Research Institute"
        dataset.source = "Regional Climate Model v2.1"
        dataset.history = f"Created on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        dataset.references = "Climate Model Documentation v2.1"
        dataset.comment = "Demonstration dataset for NetCDF4 CF conventions"
        
        # Create dimensions
        time_dim = dataset.createDimension('time', None)  # Unlimited for climate data
        lat_dim = dataset.createDimension('lat', 25)
        lon_dim = dataset.createDimension('lon', 40)
        level_dim = dataset.createDimension('level', 10)
        
        print(f"Created dimensions: time (unlimited), lat (25), lon (40), level (10)")
        
        # Create coordinate variables with CF-compliant metadata
        times = dataset.createVariable('time', 'f8', ('time',))
        times.standard_name = 'time'
        times.long_name = 'time'
        times.units = 'days since 1950-01-01 00:00:00'
        times.calendar = 'gregorian'
        times.axis = 'T'
        
        latitudes = dataset.createVariable('lat', 'f8', ('lat',))
        latitudes.standard_name = 'latitude'
        latitudes.long_name = 'latitude'
        latitudes.units = 'degrees_north'
        latitudes.axis = 'Y'
        
        longitudes = dataset.createVariable('lon', 'f8', ('lon',))
        longitudes.standard_name = 'longitude'
        longitudes.long_name = 'longitude' 
        longitudes.units = 'degrees_east'
        longitudes.axis = 'X'
        
        levels = dataset.createVariable('level', 'f8', ('level',))
        levels.standard_name = 'atmosphere_pressure_coordinate'
        levels.long_name = 'pressure level'
        levels.units = 'hPa'
        levels.positive = 'down'
        levels.axis = 'Z'
        
        # Write coordinate data
        # Time: monthly data for 2 years
        base_date = datetime(1950, 1, 1)
        start_date = datetime(2023, 1, 1)
        days_since_base = (start_date - base_date).days
        times[:24] = [days_since_base + i * 30 for i in range(24)]  # Monthly intervals
        
        latitudes[:] = np.linspace(25, 65, 25)  # Northern mid-latitudes
        longitudes[:] = np.linspace(-125, -65, 40)  # North America longitude range
        levels[:] = np.array([1000, 925, 850, 700, 600, 500, 400, 300, 250, 200])  # Standard pressure levels
        
        print(f"Coordinate ranges:")
        print(f"  Time: {times[0]:.0f} to {times[23]:.0f} days since 1950-01-01")
        print(f"  Latitude: {latitudes[0]:.1f}° to {latitudes[-1]:.1f}°N")
        print(f"  Longitude: {longitudes[0]:.1f}° to {longitudes[-1]:.1f}°E")
        print(f"  Pressure levels: {levels[0]:.0f} to {levels[-1]:.0f} hPa")
        
        # Create climate variables with full CF metadata
        print(f"\nCreating climate variables:")
        
        # Temperature
        temperature = dataset.createVariable('ta', 'f4', ('time', 'level', 'lat', 'lon'),
                                           zlib=True, complevel=4, shuffle=True, 
                                           fill_value=1.e20)
        temperature.standard_name = 'air_temperature'
        temperature.long_name = 'Air Temperature'
        temperature.units = 'K'
        temperature.coordinates = 'time level lat lon'
        temperature.grid_mapping = 'crs'
        temperature.cell_methods = 'time: mean'
        
        # Specific humidity
        humidity = dataset.createVariable('hus', 'f4', ('time', 'level', 'lat', 'lon'),
                                        zlib=True, complevel=4, shuffle=True,
                                        fill_value=1.e20)
        humidity.standard_name = 'specific_humidity'
        humidity.long_name = 'Specific Humidity'
        humidity.units = 'kg kg-1'
        humidity.coordinates = 'time level lat lon'
        humidity.grid_mapping = 'crs'
        humidity.cell_methods = 'time: mean'
        
        # Surface pressure
        surface_pressure = dataset.createVariable('ps', 'f4', ('time', 'lat', 'lon'),
                                                zlib=True, complevel=4, shuffle=True,
                                                fill_value=1.e20)
        surface_pressure.standard_name = 'surface_air_pressure'
        surface_pressure.long_name = 'Surface Air Pressure'
        surface_pressure.units = 'Pa'
        surface_pressure.coordinates = 'time lat lon'
        surface_pressure.grid_mapping = 'crs'
        surface_pressure.cell_methods = 'time: mean'
        
        # Precipitation
        precipitation = dataset.createVariable('pr', 'f4', ('time', 'lat', 'lon'),
                                             zlib=True, complevel=4, shuffle=True,
                                             fill_value=1.e20)
        precipitation.standard_name = 'precipitation_flux'
        precipitation.long_name = 'Precipitation'
        precipitation.units = 'kg m-2 s-1'
        precipitation.coordinates = 'time lat lon'
        precipitation.grid_mapping = 'crs'
        precipitation.cell_methods = 'time: mean'
        
        # Coordinate reference system
        crs = dataset.createVariable('crs', 'i4')
        crs.grid_mapping_name = 'latitude_longitude'
        crs.longitude_of_prime_meridian = 0.0
        crs.semi_major_axis = 6378137.0
        crs.inverse_flattening = 298.257223563
        
        print(f"  Created variables: ta, hus, ps, pr, crs")
        
        # Generate realistic climate data
        print(f"\nGenerating realistic climate data:")
        
        for t in range(24):  # 24 months
            month = (t % 12) + 1
            
            # Temperature with seasonal cycle and altitude dependence
            for k, pressure in enumerate(levels[:]):
                # Temperature decreases with altitude (lapse rate ~6.5K/km)
                altitude_km = (1013.25 - pressure) / 100  # Rough altitude estimate
                base_temp = 288.15 - 6.5 * altitude_km  # Standard atmosphere
                
                # Seasonal cycle
                seasonal_factor = 15 * np.cos(2 * np.pi * (month - 7) / 12)  # Max in July
                
                # Latitude gradient
                lat_factor = -20 * (latitudes[:] - 45) / 20  # Cooler toward poles
                
                temp_noise = np.random.normal(0, 2, size=(25, 40))
                temperature_slice = base_temp + seasonal_factor + lat_factor[:, None] + temp_noise
                temperature[t, k, :, :] = temperature_slice

                # Specific humidity (decreases with altitude and temperature). Computed from the slice
                # in memory rather than read back from the compressed variable.
                sat_mixing_ratio = (
                    0.622 * 6.112
                    * np.exp(17.67 * (temperature_slice - 273.15) / (temperature_slice - 29.65))
                    / pressure
                )
                relative_humidity = 0.4 + 0.4 * np.random.random(size=(25, 40))  # 40-80% RH
                humidity[t, k, :, :] = sat_mixing_ratio * relative_humidity
            
            # Surface pressure with realistic variation
            base_pressure = 101325.0  # Pa
            altitude_effect = -12.0 * (latitudes[:] - 25)  # Higher pressure at lower latitudes
            synoptic_variation = np.random.normal(0, 500, size=(25, 40))  # Weather systems
            surface_pressure[t, :, :] = base_pressure + altitude_effect[:, None] + synoptic_variation
            
            # Precipitation with seasonal and spatial patterns
            for j in range(25):
                for i in range(40):
                    # Seasonal precipitation (more in summer/winter depending on location)
                    seasonal_precip = 1 + 0.5 * np.sin(2 * np.pi * (month - 3) / 12)
                    
                    # Latitude effect (more precipitation at mid-latitudes)
                    lat_effect = 1.5 - 0.5 * np.abs(latitudes[j] - 45) / 20
                    
                    # Random weather
                    base_precip = 2e-6  # kg m-2 s-1 (about 2mm/day average)
                    precip_noise = np.random.exponential(1.0)
                    
                    precipitation[t, j, i] = base_precip * seasonal_precip * lat_effect * precip_noise
        
        print(f"  Climate data generated for 24 months")
        
        # Add data quality and processing information
        print(f"\nAdding data quality metadata:")
        
        # Variable attributes for data quality
        for var in [temperature, humidity, surface_pressure, precipitation]:
            var.missing_value = 1.e20
            var.valid_range = [-1.e10, 1.e10]
            
        # Processing-specific attributes
        temperature.ancillary_variables = "ta_quality_flag"
        humidity.ancillary_variables = "hus_quality_flag" 
        
        # Create quality flag variables (simplified)
        temp_qc = dataset.createVariable('ta_quality_flag', 'i2', ('time', 'level', 'lat', 'lon'),
                                       zlib=True, fill_value=-1)
        temp_qc.long_name = 'Air Temperature Quality Flag'
        temp_qc.flag_values = [0, 1, 2, 3]
        temp_qc.flag_meanings = "good suspect_data bad_data missing_data"
        
        # Set all data as good quality for demonstration
        temp_qc[:] = 0
        
        print(f"  Quality control metadata added")
    
    # Demonstrate CF compliance checking
    print(f"\nCF compliance demonstration:")
    
    with nc.Dataset(climate_file, 'r') as dataset:
        # Check CF compliance elements
        cf_elements = {
            'Conventions': getattr(dataset, 'Conventions', 'Not specified'),
            'Standard names': 'All variables use CF standard names',
            'Units': 'All variables use UDUNITS-compatible units',
            'Coordinates': 'All data variables specify coordinate variables',
            'Grid mapping': 'Coordinate reference system specified',
            'Cell methods': 'Temporal averaging methods specified'
        }
        
        print(f"CF compliance elements:")
        for element, description in cf_elements.items():
            print(f"  ✓ {element}: {description}")
        
        # Demonstrate metadata access
        print(f"\nMetadata access examples:")
        
        temp_var = dataset.variables['ta']
        print(f"  Temperature metadata:")
        print(f"    Standard name: {temp_var.standard_name}")
        print(f"    Units: {temp_var.units}")
        print(f"    Shape: {temp_var.shape}")
        print(f"    Cell methods: {temp_var.cell_methods}")
        
        # Data access examples
        print(f"\nData access examples:")
        
        # Surface temperature for first month
        surface_temp = temp_var[0, -1, :, :]  # Surface level (highest pressure)
        print(f"  Surface temperature (first month): {surface_temp.min():.1f} to {surface_temp.max():.1f} K")
        
        # Vertical profile at central location
        central_profile = temp_var[0, :, 12, 20]  # Central lat/lon
        print(f"  Vertical temperature profile: {central_profile.min():.1f} to {central_profile.max():.1f} K")
        
        # Time series at surface
        surface_timeseries = temp_var[:, -1, 12, 20]
        print(f"  Surface temperature time series: {surface_timeseries.min():.1f} to {surface_timeseries.max():.1f} K")
    
    # CF conventions benefits
    print(f"\nBenefits of CF conventions:")
    
    cf_benefits = [
        "Interoperability: Standardized metadata enables tool compatibility",
        "Data discovery: Consistent naming facilitates dataset search",
        "Automatic processing: Tools can interpret data without manual configuration",
        "Quality assurance: Standard practices improve data reliability",
        "Long-term preservation: Ensures data remains accessible over time",
        "Community standards: Facilitates collaboration and data sharing"
    ]
    
    for benefit in cf_benefits:
        print(f"  - {benefit}")
    
    # Common CF-compliant tools
    print(f"\nTools supporting CF conventions:")
    
    cf_tools = [
        "xarray: Python library with native CF support",
        "NCO: NetCDF Operators for file manipulation",
        "CDO: Climate Data Operators for analysis", 
        "Panoply: NASA tool for NetCDF data visualization",
        "Ferret: NOAA analysis and visualization package",
        "THREDDS: Web-based data access and catalogs"
    ]
    
    for tool in cf_tools:
        print(f"  - {tool}")
    
    # File size and performance
    file_size = os.path.getsize(climate_file)
    print(f"\nDataset statistics:")
    print(f"  File size: {file_size / 1024 / 1024:.1f} MB")
    print(f"  Data points: {24 * 10 * 25 * 40:,} (4D temperature)")
    print(f"  Compression: ~{100 * (1 - file_size / (24 * 10 * 25 * 40 * 4 * 4)):.1f}% space savings")
    
    # Clean up
    os.unlink(climate_file)
    print(f"\nClimate dataset file cleaned up")
    
    print("\\nClimate data workflows and CF conventions completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    print("Demonstrating CF conventions concepts:")
    concepts = [
        "CF compliance: Following Climate and Forecast metadata conventions",
        "Standard names: Using standardized variable nomenclature",
        "Metadata completeness: Comprehensive variable and dataset descriptions"
    ]
    for concept in concepts:
        print(f"  - {concept}")
    raise

print("Example 5 finished")